<a href="https://colab.research.google.com/github/FarabiOnAMission/Machine-Learning-Stuffs/blob/main/Naive_Bayes_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
import math

class NaiveBayes:

  def __init__(self,smoothing=1.0):
    self.smoothing = smoothing
    self.class_counts = {} #Total Number of Class Counts like Spam : 5, Not Spam: 10
    self.word_counts = {} #how many time a specific word appeared in this class
    self.total_words = {} #Total Number of words in this
    self.vocab = [] #master list containing each unique word that appeared in this class
    self.length_counts = {} #New Feature that contains the sentence lengths

  def train(self, documents, labels):
    #documents = ["win free money now", "hey call me tomorrow", "cheap medicine online", ...]
    #labels = ["spam", "ham", "spam" , ....]
    for i in range(len(documents)):
      doc = documents[i]
      label = labels[i]
      if label not in self.class_counts:
        self.class_counts[label] = 0 #intializes a class like spam with 0 count
        self.word_counts[label] = {} #creates its own dictionary which contains all words
        self.total_words[label] = 0 #creates a count for the total number of words, later needed in probability
        self.length_counts[label] = {"short":0, "long":0} #two lengths and counter for each of them for a specific label

        #logic for handling the sentence length
        if(len(doc)<100):
          doc_length_cat = "short"
        else:
          doc_length_cat = "long"

      self.length_counts[label][doc_length_cat] += 1

      self.class_counts[label] += 1 #increments the class count by 1
      words = doc.lower().split() #splits all the words here and converts it to lower
      for word in words: #iterates through all the words
        if word not in self.word_counts[label]: #if a word found which is not yet in the dictionary of the current label
          self.word_counts[label][word] = 0 #then creates this word count under this label with 0
        self.word_counts[label][word] += 1 #increments the word
        self.total_words[label] += 1 #increments the total_count for this label
        if word not in self.vocab:
          self.vocab.append(word) #appends the unknown word to all possible known words

  def predict(self, document): #after trainining the dataset, now this function is used for predicting
    words = document.lower().split() #gets a new document and now splits that document word by word
    total_docs = 0 #initializs a variable to track how many emails you looked at during training
    for count in self.class_counts.values():
      total_docs += count
    vocab_size = len(self.vocab) #how many words you looked during training

    if len(document) <100:
      new_doc_length = "short"
    else:
      new_doc_length = "long"

    #setting up the competition
    best_class = None
    best_score = -99999999

    for cls in self.class_counts:
      score = math.log(self.class_counts[cls] / total_docs) #Setting up the prior, head_start based on previous datasets

      len_count = self.length_counts[cls][new_doc_length]
      total_cls_docs = self.class_counts[cls]

      len_probability = math.log(len_count + self.smoothing)/(total_cls_docs + self.smoothing*2)
      score+= math.log(len_probability)

      for word in words:
      #looks at a word and then finds how many times this word appeared in our spam,not spam etc class
        if word not in self.word_counts[cls]:
          count = 0
        else:
          count = self.word_counts[cls][word]
        total = self.total_words[cls] #finds the total number of words in this class
        score += math.log((count + self.smoothing) / (total + self.smoothing * vocab_size)) #applies the probability formula and adds it to our prior score
      #why did we add vocab size, because we added smoothing to each of our unique word to avoid overflow or underflow

      if score > best_score:
        best_class = cls
        best_score = score
    return best_class

In [13]:
train_docs = [
    "win free money now",
    "free lottery ticket winner",
    "claim your prize today free",
    "urgent offer free cash",
    "congratulations you won free",
    "meeting tomorrow at noon",
    "project update attached",
    "can we schedule a call",
    "quarterly report review",
    "lunch on thursday sounds good",
    "team standup notes attached",
    "please review the pull request",
]

train_labels = [
    "spam", "spam", "spam", "spam", "spam",
    "ham", "ham", "ham", "ham", "ham", "ham", "ham",
]

classifier = NaiveBayes()
classifier.train(train_docs,train_labels)

In [14]:
test_messages = [
    "free money waiting for you",
    "meeting rescheduled to friday",
    "you won a free prize",
    "please review the attached report",
]

for msg in test_messages:
  print(f"{msg}: {classifier.predict(msg)}")

free money waiting for you: spam
meeting rescheduled to friday: ham
you won a free prize: spam
please review the attached report: ham


In [15]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_docs)
clf = MultinomialNB()
clf.fit(X_train, train_labels)

X_test = vectorizer.transform(test_messages)
predictions = clf.predict(X_test)

for msg, pred in zip(test_messages, predictions):
    print(f"  '{msg}' -> {pred}")

  'free money waiting for you' -> spam
  'meeting rescheduled to friday' -> ham
  'you won a free prize' -> spam
  'please review the attached report' -> ham
